# Lab | BabyAGI with agent

> **Nota:** Este lab ha sido reescrito con la API moderna de LangChain (LCEL).
> Las clases y funciones deprecadas que se han reemplazado son:
> - `LLMChain` → LCEL `prompt | llm | StrOutputParser()`
> - `ZeroShotAgent` + `ZeroShotAgent.create_prompt` → `create_react_agent` con prompt personalizado
> - `AgentExecutor.from_agent_and_tools` → `AgentExecutor(agent=..., tools=...)`
> - `FAISS(embed_fn, ...)` → `FAISS(embedding_function=embeddings_model, ...)`
> - `Chain.__call__({...})` → `.invoke({...})`

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats.
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-key) to run this lab.

## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information.

## Install and Import Required Modules

In [1]:
!pip install -qU langchain langchain_community langchain_experimental langchain_openai faiss-cpu google-search-results


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import Optional

# Importaciones modernas — LLMChain está deprecado
# Antes (DEPRECADO): from langchain.chains import LLMChain
# Ahora (LCEL): se usa prompt | llm | StrOutputParser()
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

c:\Users\jesur\Desktop\ACurso\w7\.w7\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\jesur\Desktop\ACurso\w7\.w7\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
import faiss
from langchain_community.vectorstores import FAISS
# InMemoryDocstore actualizado a langchain_community
# Antes: from langchain.docstore import InMemoryDocstore
from langchain_community.docstore.in_memory import InMemoryDocstore

In [8]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [5]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)

# Forma moderna de inicializar FAISS:
# Antes (DEPRECADO — pasaba una función en lugar de un objeto Embeddings):
# vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})
# Ahora:
vectorstore = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [11]:
# Importaciones modernas para agentes
# Antes (DEPRECADO): from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
# Ahora:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_classic import hub
from langchain_community.utilities import SerpAPIWrapper
from langchain_core.tools import Tool

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# --- TODO chain ---
# Antes (DEPRECADO): todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
# Ahora (LCEL):
todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. "
    "Come up with a todo list for this objective: {objective}"
)
todo_chain = todo_prompt | llm | StrOutputParser()

# Los Tool esperan una función str → str.
# Los LCEL chains usan .invoke(); envolvemos con una lambda.
# Antes (DEPRECADO): func=todo_chain.run
def todo_fn(objective: str) -> str:
    return todo_chain.invoke({"objective": objective})

search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_fn,
        description="useful for when you need to come up with todo lists. "
                    "Input: an objective to create a todo list for. "
                    "Output: a todo list for that objective. "
                    "Please be very clear what the objective is!",
    ),
]

In [13]:
# Prompt personalizado para create_react_agent compatible con BabyAGI.
# BabyAGI pasa {objective}, {context} y {task} al AgentExecutor,
# además de {tools}, {tool_names} y {agent_scratchpad} que necesita ReAct.
#
# Antes (DEPRECADO):
# prompt = ZeroShotAgent.create_prompt(
#     tools,
#     prefix=prefix, suffix=suffix,
#     input_variables=["objective", "task", "context", "agent_scratchpad"],
# )
# llm_chain = LLMChain(llm=llm, prompt=prompt)
# agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
#
# Ahora (LCEL):
BABY_AGI_REACT_TEMPLATE = """You are an AI who performs one task based on the following objective: {objective}.
Take into account these previously completed tasks: {context}.

You have access to the following tools:

{tools}

Use the following format:

Question: the input task you must complete
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input task

Begin!

Question: {task}
{agent_scratchpad}"""

react_prompt = PromptTemplate(
    input_variables=["objective", "context", "task", "tools", "tool_names", "agent_scratchpad"],
    template=BABY_AGI_REACT_TEMPLATE,
)

# create_react_agent reemplaza a ZeroShotAgent
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=react_prompt)

# AgentExecutor(agent=...) reemplaza a AgentExecutor.from_agent_and_tools(...)
# Antes (DEPRECADO): AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [14]:
OBJECTIVE = "Write a weather report for SF today"

In [15]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain  # or from langchain.chains import LLMChain

# Define your execution prompt (example)
execution_template = """You are an AI who performs one task based on the following objective: {objective}.
Your task: {task}
Context: {context}

Response:"""

execution_prompt = PromptTemplate(
    template=execution_template,
    input_variables=["objective", "context", "task"]
)

# ← THIS IS THE IMPORTANT PART
execution_chain = LLMChain(
    llm=llm,
    prompt=execution_prompt,
    output_key="result",          # ← Add this line
    verbose=True
)

C:\Users\jesur\AppData\Local\Temp\ipykernel_28508\2453727707.py:17: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  execution_chain = LLMChain(


In [19]:
from langchain_experimental.autonomous_agents import BabyAGI

baby_agi = BabyAGI.from_llm(
    llm=llm,
    task_execution_chain=execution_chain,   # make sure you're passing the fixed chain
    vectorstore=your_vectorstore,           # whatever you're using
    verbose=True,
    max_iterations=5
)

result = baby_agi.invoke({"objective": OBJECTIVE})

NameError: name 'your_vectorstore' is not defined

In [21]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_experimental.autonomous_agents import BabyAGI

# ====================== Setup ======================
os.environ["OPENAI_API_KEY"] = "sk-..."   # ← Put your key here

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
embeddings = OpenAIEmbeddings()

# ====================== Execution Chain ======================
execution_template = """You are an AI who performs one task based on the following objective: {objective}.
Your task: {task}
Context: {context}

Response:"""

execution_prompt = PromptTemplate(
    template=execution_template,
    input_variables=["objective", "context", "task"]
)

execution_chain = LLMChain(
    llm=llm,
    prompt=execution_prompt,
    output_key="result",
    verbose=False
)

# ====================== Vectorstore with proper metadata ======================
# Initial task with correct metadata structure that BabyAGI expects
initial_document = Document(
    page_content="Initial task: Understand the objective and create first tasks",
    metadata={"task": "Understand the objective and create first tasks"}
)

vectorstore = FAISS.from_documents([initial_document], embeddings)

# ====================== BabyAGI ======================
OBJECTIVE = "Write a short blog post about the benefits of AI in education"

baby_agi = BabyAGI.from_llm(
    llm=llm,
    task_execution_chain=execution_chain,
    vectorstore=vectorstore,
    verbose=True,
    max_iterations=6
)

# Run it
result = baby_agi.invoke({"objective": OBJECTIVE})
print(result)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}